### simple parallel workflow : 

In [53]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict

In [54]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours : int 
    sixs : int 

    sr : float 
    ball_per_boundary : float 
    boundary_percentage : float 
    summary : str

In [55]:
# we are partially updating the state in parallel and then we will use the updated state to generate the summary.

def calculate_sr(state : BatsmanState) :
    runs = state['runs']
    balls = state['balls']
    # it is updatinig the state with the new value of sr which is calculated using runs and balls.
    # as we returning a dictionary with the key as sr and value as the calculated sr, the state will be updated with this new value of sr.
    return {'sr' : runs/balls*100}

def calculate_balls_per_boundary(state : BatsmanState) :
    balls = state['balls']
    fours = state['fours']
    sixs = state['sixs']
    return {'ball_per_boundary' : balls/(fours + sixs)}

def calculate_boundary_percentage(state : BatsmanState) :
    bp = (state['balls']*4 + state['balls']*6)/state['runs']*100
    return {'boundary_percentage' : bp}


def summary(state : BatsmanState) :
    summary = f"""
    Strike Rate : {state['sr']}
    Balls : {state['balls']}
    Balls per Boundary : {state['ball_per_boundary']}
    Boundary Percentage : {state['boundary_percentage']}
"""
    
    return {'summary' : summary}

In [56]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr' , calculate_sr)
graph.add_node('calculate_balls_per_boundary' , calculate_balls_per_boundary)
graph.add_node('calculate_boundary_percentage' , calculate_boundary_percentage)
graph.add_node('summary' , summary)


In [57]:
# we will define three edges at the same time for parallel 
graph.add_edge(START , 'calculate_sr' )
graph.add_edge(START , 'calculate_balls_per_boundary' )
graph.add_edge(START , 'calculate_boundary_percentage' )

graph.add_edge('calculate_sr' , 'summary')
graph.add_edge('calculate_balls_per_boundary' , 'summary')
graph.add_edge('calculate_boundary_percentage' , 'summary')

graph.add_edge('summary' , END)

workflow = graph.compile()

In [58]:
initial_state = {
    'runs' : 100,
    'balls' : 60,
    'fours' : 10,
    'sixs' : 5
}
workflow.invoke(initial_state)
# but this will not work as the three nodes are running in parallel and they are trying to update the same state at the same time which is not possible. We need to use locks to ensure that only one node is updating the state at a time.
# rather than standing the entire state we can just return the value that we want to update and then we can merge the values in the summary node. This way we can avoid the need for locks and we can run the nodes in parallel without any issues.



{'runs': 100,
 'balls': 60,
 'fours': 10,
 'sixs': 5,
 'sr': 166.66666666666669,
 'ball_per_boundary': 4.0,
 'boundary_percentage': 600.0,
 'summary': '\n    Strike Rate : 166.66666666666669\n    Balls : 60\n    Balls per Boundary : 4.0\n    Boundary Percentage : 600.0\n'}

### let's continue 

## Reducers ? 


it defines How state should be updated when multiple nodes write to the same key.
In LangGraph:

Multiple nodes can update the same state
Updates can happen in parallel or sequentially

👉 So LangGraph needs a rule:

“Should I overwrite, append, merge, or combine?”

In [73]:
# a simple workflow where we will ask question to llm and it will ans  and then we will end .
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict , Annotated
from dotenv import load_dotenv
from pydantic import BaseModel , Field
import os
import operator
load_dotenv()


True

In [93]:
model = ChatGoogleGenerativeAI(api_key=os.getenv("GEMINI_API_KEY") , model="gemini-3-flash-preview")

In [84]:
class EvaluationSchema(BaseModel):
    feedback : str = Field(description="detailed Feedback for the assay")
    score : int = Field(description="score out of 10 for the assay" , ge=0 , le=10)
    
    

In [85]:
structered_model = model.with_structured_output(EvaluationSchema)
# this model will return the output in the form of EvaluationSchema which is a pydantic model with two fields feedback and score.
# so this will give us the structured output only . 


In [92]:
class UPSCState(TypedDict):
    essay : str 
    language_feedback : str 
    analysis_feedback : str
    clarity_feedback : str
    overall_feedback : str  
    # individual_scores : list[int]
    # now if all three feedback are generated by llm in parallel . 
    # if it tries to change the state at the same time then it will create a conflict and we will not get the correct output.
    # so we have to merge the outputs of all three feedbacks in the end and then update the state with the merged output. This way we can avoid the conflict and we can run the nodes in parallel without any issues.
    # we will do this using operator 
    # which is a python module that provides a set of functions that can be used to perform various operations on data . 
    individual_scores : Annotated[list[int], operator.add]
    # each feedback will return a score(in a list) out of 10 and we want to add all the scores to get the total score out of 30. So we can use operator.add to add the scores together and get the total score.
    # here we are basically using a reducer function to reduce the list of scores to a single score by adding them together
    avg_score : float
    



In [87]:
def language(state : UPSCState) :
    feedback = structered_model.invoke(f"Give detailed feedback on the language quality used in the following essay : {state['essay']}")

    return { 
        'language_feedback' : feedback.feedback,
        'individual_scores' : [feedback.score]
    }
def analysis(state : UPSCState) :
    feedback = structered_model.invoke(f"Give detailed feedback on the depth of analysis used in the following essay : {state['essay']}")

    return { 
        'analysis_feedback' : feedback.feedback,
        'individual_scores' : [feedback.score]
    }
def clarity(state : UPSCState) :
    feedback = structered_model.invoke(f"Give detailed feedback on the clarity of thought used in the following essay : {state['essay']}")

    return { 
        'clarity_feedback' : feedback.feedback,
        'individual_scores' : [feedback.score]
    }

def overall_feedback(state : UPSCState) :
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content[0]['text']  
    # avg calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])
    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}
    

In [88]:
graph = StateGraph(UPSCState)

graph.add_node('language_feedback' , language)
graph.add_node('analysis_feedback' , analysis)
graph.add_node('clarity_feedback' , clarity)
graph.add_node('overall_feedback' , overall_feedback)

In [89]:
graph.add_edge(START , 'language_feedback' )    
graph.add_edge(START , 'analysis_feedback' )
graph.add_edge(START , 'clarity_feedback' )
graph.add_edge('language_feedback' , 'overall_feedback')
graph.add_edge('analysis_feedback' , 'overall_feedback')
graph.add_edge('clarity_feedback' , 'overall_feedback')
graph.add_edge('overall_feedback' , END)
workflow = graph.compile()

In [90]:
essay = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""

In [91]:
workflow.invoke({
    'essay' : essay})

{'essay': 'India and AI Time\n\nNow world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.\n\nIndia have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.\n\nIn farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.\n\nBut problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.\n\nOne more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.\n\n